# M6 on Kaggle: full-pool incremental learning and rehearsal

**Attach as input:** the same private Kaggle dataset as `m3_kaggle.ipynb` (the five files of the M1 `processed/` folder).

**Settings:** Internet *On*. Accelerator: *None*. M6 fits 15 clients per window on CPU.
**Run:** *Save Version → Save & Run All (Commit)* so it keeps running with the browser closed.
Time is an estimate, not a measurement: the dev run took 15 min at 4 threads, and scaling that by row count gives about 4 hours.

**Output:** `/kaggle/working/m6_full_outputs.zip` contains `reports/m6/full/` and the run folder. Download it and unzip it at the repo root.


In [ ]:
import os, subprocess, sys
REPO = "/kaggle/working/IDS-code"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Indrajit147/IDS-code.git", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
# --no-deps: keep Kaggle's preinstalled stack intact; versions are recorded per run.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO, "--no-deps"], check=True)
os.chdir(REPO)
os.environ["IDS_REPO_ROOT"] = REPO
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# Rebuild the M2 partition for the full pool (deterministic, ~1 min)
!python scripts/m2_build_partitions.py --env kaggle --pools full
# M2 redraws these figures, and their bytes differ on Kaggle. Nothing reads them. A changed report or JSON stays flagged.
!git checkout -- 'reports/m2/full/*.png'


In [ ]:
# 4 threads, matching the dev run, because results reproduce exactly only at the same thread count
!python scripts/m6_incremental.py --env kaggle --pool full --set resources.n_jobs=4


In [ ]:
import shutil, glob, os
stage = "/kaggle/working/m6_stage"
shutil.rmtree(stage, ignore_errors=True)
shutil.copytree("reports/m6/full", f"{stage}/reports/m6/full")
runs = sorted(glob.glob("/kaggle/working/results/runs/*_m6_incremental_full"))
if runs:
    shutil.copytree(runs[-1], f"{stage}/results/runs/{os.path.basename(runs[-1])}")
shutil.make_archive("/kaggle/working/m6_full_outputs", "zip", stage)
shutil.rmtree(stage)
print(open("reports/m6/full/INCREMENTAL.md", encoding="utf-8").read()[:4000])
